# 📊 INF01090 - Ciência de Dados - Regression Techniques

**House Prices - Advanced Regression Techniques - Kaggle-Style Competition**

House Prices – Advanced Regression Techniques (<https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/>) is a long-running Kaggle competition that challenges participants to predict the sale prices of homes in Ames, Iowa, using advanced regression models. It is one of Kaggle’s most popular “Getting Started” challenges, designed to build practical skills in data cleaning, feature engineering, and predictive modelling.

## Key facts

- **Platform:** Kaggle
- **Launch year:** 2016
- **Original dataset:** 1,460 training and 1,459 test homes
- **Features:** 79 explanatory variables
- **Primary goal:** Predict house sale price


## 📂 Dataset and Objective

The dataset, compiled by Dean De Cock as an update to the classic Boston Housing dataset, describes almost every aspect of residential homes—covering lot size, room counts, materials, neighbourhood, and more. The task is to use these features to predict each property's final sale price, making it a supervised regression problem that blends statistical and machine learning techniques.

The file **`data_description.txt`** has the full description of each column.

## 🛠 Skills and techniques

This competition is widely used to practise:

- **Feature engineering:** handling missing data, encoding categorical variables, transforming skewed features
- **Model building:** experimenting with algorithms such as linear regression, ridge, lasso, random forest, and gradient boosting
- **Evaluation:** models are ranked by **RMSE on log-transformed prices** in our local competition setup


## 🎯 Assignment Goal

Your goal is to build a regression pipeline that predicts **`SalePrice`** for the hidden test set.

This is not only a leaderboard exercise. You should use this assignment to demonstrate that you understand:

- data cleaning for tabular data
- feature encoding and transformation
- regression modeling
- error analysis
- the effect of different design choices on predictive performance


## 📁 Files You Will Receive

You should work only with the files distributed for this lab:

- **`train_student.csv`** — training data with the target column
- **`test_student.csv`** — test data without the target column
- **`submission_template.csv`** — expected format for submission
- **`data_description.txt`** — attribute descriptions

Do **not** use Kaggle's original public test split for submission. The grading app uses a **custom hidden split** created for this class.


## 🏁 Submission and Leaderboard

Submissions are evaluated in the local grading app:

<https://labo5inf01090-huzhzhojtbeqqknm7duabo.streamlit.app/>

The app expects a CSV file with exactly these columns:

```csv
Id,prediction
1461,210000
1462,179500
1463,220000
```

Rules:

- `Id` must match the IDs in **`test_student.csv`**
- `prediction` must contain one numeric prediction per row
- all test rows must be present
- predictions for `SalePrice` should be non-negative


## 📌 What You Must Deliver

Each group must submit:

1. **A prediction file** for the leaderboard  
2. **This notebook** (completed, with code, outputs, and short explanations)  
3. **A short report section in the notebook** explaining:
   - preprocessing choices
   - feature engineering
   - model(s) tested
   - final model used
   - interpretation of the obtained score


## 👥 Group Work

- Work in groups of up to 4
- All members of the group must understand the final solution
- Use a consistent team name in the leaderboard
- The same team may submit multiple times; the leaderboard keeps the best score


## 📊 Grading Criteria

Your grade will not depend only on leaderboard position. The first three places will get additional grade.

Important:
- a top leaderboard score with poor documentation is **not enough**
- a strong notebook with solid methodology can still receive a high grade even if it is not the top-ranked solution


## 🚦 Recommended Workflow

A good workflow for this assignment is:

1. Inspect the training data
2. Identify numeric and categorical variables
3. Handle missing values
4. Encode categorical variables
5. Optionally transform skewed variables
6. Build a baseline regression model
7. Evaluate improvements using validation on the training set
8. Train your final model on the full student training set
9. Predict on `test_student.csv`
10. Submit the predictions to the leaderboard


## ⚠️ Restrictions and Good Practice

- Do not manually inspect or reconstruct the hidden target values
- Do not hard-code predictions
- Do not submit malformed files to probe the scorer
- Do not use the leaderboard as your only validation method

Recommended:
- create your own validation split from `train_student.csv`
- compare models locally before submitting
- submit only meaningful improvements


## 🧪 Suggested Experiments

You may explore ideas such as:

- dropping columns with many missing values
- imputing missing values numerically and categorically
- one-hot encoding categorical variables
- applying `log1p(SalePrice)` during training
- trying different regularization strengths
- comparing linear and non-linear models
- checking whether some features are highly skewed


## 🧭 Starter Checklist

Before your first submission, verify that:

- [ ] `train_student.csv` loads correctly
- [ ] `test_student.csv` has the same predictor columns as expected
- [ ] your preprocessing works for both train and test
- [ ] your model produces one prediction per test row
- [ ] the output file has exactly two columns: `Id`, `prediction`
- [ ] all predictions are numeric
- [ ] all predictions are non-negative


## 🐍 Suggested Notebook Structure

You may organize your work using sections such as:

1. Data loading
2. Exploratory inspection
3. Missing-value handling
4. Feature encoding
5. Train/validation split
6. Baseline model
7. Improved model
8. Final training and test prediction
9. Submission file generation
10. Reflection


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge


## 1. Load the data

Update the paths if necessary.


In [2]:
train_df = pd.read_csv("train_student.csv")
test_df = pd.read_csv("test_student.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


Train shape: (1022, 81)
Test shape: (438, 80)


## 2. Identify target and predictors


In [10]:
target_col = "SalePrice"
id_col = "Id"

X = train_df.drop(columns=[target_col])
y = train_df[target_col].copy()

print("Target summary:")
display(y.describe())


Target summary:


count      1022.000000
mean     181312.692759
std       77617.461005
min       34900.000000
25%      130000.000000
50%      165000.000000
75%      215000.000000
max      745000.000000
Name: SalePrice, dtype: float64

### 2.1 Missing-value handling



In [5]:
# Null values analysis
numeric_missing = X.select_dtypes(include=["number"]).isnull().sum()
print("Numeric features with missing values:")
print(numeric_missing[numeric_missing > 0])

categorical_missing = X.select_dtypes(exclude=["number"]).isnull().sum()
print("Categorical features with missing values:")
print(categorical_missing[categorical_missing > 0])

print("duplicated rows:", X.duplicated().sum())


Numeric features with missing values:
LotFrontage    190
MasVnrArea       3
GarageYrBlt     54
dtype: int64
Categorical features with missing values:
Alley            956
MasVnrType       590
BsmtQual          26
BsmtCond          26
BsmtExposure      26
BsmtFinType1      26
BsmtFinType2      26
Electrical         1
FireplaceQu      487
GarageType        54
GarageFinish      54
GarageQual        54
GarageCond        54
PoolQC          1017
Fence            820
MiscFeature      982
dtype: int64
duplicated rows: 0


### 2.2 Checking Outliers

In [12]:
#Check outliers
quartiles = X[X.select_dtypes(include=["number"]).columns].quantile([0.01, 0.99])   
print(quartiles)

#See which values pass the quartiles
for col in X.select_dtypes(include=["number"]).columns:
    print(f"Values in {col} below the 1st percentile:")
    print(X[X[col] < quartiles.loc[0.01, col]][col])
    print(f"Values in {col} above the 99th percentile:")
    print(X[X[col] > quartiles.loc[0.99, col]][col])

#Excluding outliers
outlier_idx = X.loc[X["Id"].isin([1299, 524])].index
X.drop(index=outlier_idx, inplace=True)
y.drop(index=outlier_idx, inplace=True)

           Id  MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  \
0.01    12.21        20.0        21.00   1680.00         3.00          3.0   
0.99  1444.79       190.0       143.37  44443.74         9.79          9.0   

      YearBuilt  YearRemodAdd  MasVnrArea  BsmtFinSF1  ...  GarageArea  \
0.01    1893.68        1950.0        0.00        0.00  ...        0.00   
0.99    2009.00        2009.0      744.43     1572.79  ...      965.06   

      WoodDeckSF  OpenPorchSF  EnclosedPorch  3SsnPorch  ScreenPorch  \
0.01        0.00         0.00           0.00       0.00         0.00   
0.99      532.43       291.79         262.95     177.48       275.37   

      PoolArea  MiscVal  MoSold  YrSold  
0.01       0.0      0.0     1.0  2006.0  
0.99       0.0   1189.5    12.0  2010.0  

[2 rows x 37 columns]
Values in Id below the 1st percentile:
125     4
129     6
185     3
186     7
340    10
369     8
420     1
590    12
658     5
781     9
837     2
Name: Id, dtype: int64
Valu

### 2.3 Feature Encoding

In [ ]:
def safe_divide(numerator, denominator):
    """Create stable ratio features while leaving impossible ratios for imputation."""
    return (numerator / denominator.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)


def add_engineered_features(df):
    df = df.copy()

    df["HasPool"] = (df["PoolArea"].fillna(0) > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"].fillna(0) > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"].fillna(0) > 0).astype(int)
    df["HasBasement"] = (df["TotalBsmtSF"].fillna(0) > 0).astype(int)
    df["HasMiscFeature"] = (df["MiscVal"].fillna(0) > 0).astype(int)
    df["HasAlley"] = df["Alley"].notna().astype(int)
    df["HasFence"] = df["Fence"].notna().astype(int)

    df["TotalSF"] = df[["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]].fillna(0).sum(axis=1)
    df["TotalBathrooms"] = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )
    df["TotalPorchSF"] = df[["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]].fillna(0).sum(axis=1)
    df["TotalHouseSF"] = df[["TotalBsmtSF", "GrLivArea"]].fillna(0).sum(axis=1)
    df["TotalFinishedSF"] = df[["BsmtFinSF1", "BsmtFinSF2", "1stFlrSF", "2ndFlrSF"]].fillna(0).sum(axis=1)
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["GarageAge"] = df["YrSold"] - df["GarageYrBlt"]
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
    df["NewHouse"] = (df["YrSold"] == df["YearBuilt"]).astype(int)

    # These two ratio features produced the best single-model leaderboard score.
    df["SFPerRoom"] = safe_divide(df["GrLivArea"], df["TotRmsAbvGrd"])
    df["BathPerBedroom"] = safe_divide(df["TotalBathrooms"], df["BedroomAbvGr"])

    df["DecadeBuilt"] = (df["YearBuilt"] // 10) * 10
    df["DecadeRemod"] = (df["YearRemodAdd"] // 10) * 10
    df["DecadeSold"] = (df["YrSold"] // 10) * 10

    drop_columns = [
        id_col,
        "Utilities",
        "PoolQC",
        "Street",
        "MiscFeature",
        "Condition2",
        "RoofMatl",
        "Heating",
        "Exterior2nd",
        "BsmtFinType2",
        "BsmtFinSF2",
        "Alley",
        "Fireplaces",
        "FireplaceQu",
        "Fence",
        "PoolArea",
    ]
    return df.drop(columns=drop_columns, errors="ignore")

X = add_engineered_features(X)

print("Feature matrix after engineering:", X.shape)
print("Remaining categorical features:")
for col in X.select_dtypes(exclude=["number"]).columns:
    print(f"{col}: {X[col].nunique()} unique values")


## 3. Build a local validation split

Use this split to compare models **before** submitting to the leaderboard.


In [14]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Transform y to log scale for training (aligns with leaderboard evaluation)
y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)

print(X_train.shape, X_valid.shape)


(816, 79) (204, 79)


## 4. Separate numeric and categorical columns


In [10]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features)) 

Numeric features: 56
Categorical features: 31


## 5. Create a preprocessing pipeline

You may improve this pipeline as part of the assignment.


In [ ]:
class SkewLogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.75):
        self.threshold = threshold

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        skewness = X_df.skew(numeric_only=True).abs()
        candidate_cols = skewness[skewness > self.threshold].index.tolist()
        minimums = X_df[candidate_cols].min() if candidate_cols else pd.Series(dtype=float)
        self.cols_ = [col for col in candidate_cols if minimums[col] >= 0]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        if self.cols_:
            X_df[self.cols_] = np.log1p(X_df[self.cols_])
        return X_df.values


try:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("skew_log", SkewLogTransformer(threshold=0.75)),
    ("scaler", RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", onehot_encoder)
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])


## 6. Baseline model

Start with a simple model.


In [12]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

baseline_model.fit(X_train, y_train_log)
pred_valid_baseline_log = baseline_model.predict(X_valid)
pred_valid_baseline = np.maximum(np.expm1(pred_valid_baseline_log), 0)

rmse_log_baseline = mean_squared_error(y_valid_log, np.log1p(pred_valid_baseline)) ** 0.5
rmse_baseline = mean_squared_error(y_valid, pred_valid_baseline) ** 0.5
mae_baseline = mean_absolute_error(y_valid, pred_valid_baseline)
r2_baseline = r2_score(y_valid, pred_valid_baseline)

print("Baseline Log RMSE:", rmse_log_baseline)
print("Baseline RMSE    :", rmse_baseline)
print("Baseline MAE     :", mae_baseline)
print("Baseline R?      :", r2_baseline)


Baseline Log RMSE: 0.11849273881670411
Baseline RMSE    : 22136.73925043512
Baseline MAE     : 14378.638327609742
Baseline R?      : 0.9161510951064864


## 7. Improved model

Try at least one stronger model and compare the result.


In [ ]:
def make_ridge_model(alpha=16):
    return Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(alpha=alpha))
    ])

ridge_alphas = [15, 16, 17, 18]
ridge_results = []

for alpha in ridge_alphas:
    candidate_model = make_ridge_model(alpha)
    candidate_model.fit(X_train, y_train_log)

    pred_valid_log = candidate_model.predict(X_valid)
    pred_valid = np.maximum(np.expm1(pred_valid_log), 0)

    ridge_results.append({
        "alpha": alpha,
        "Log RMSE": mean_squared_error(y_valid_log, np.log1p(pred_valid)) ** 0.5,
        "RMSE": mean_squared_error(y_valid, pred_valid) ** 0.5,
        "MAE": mean_absolute_error(y_valid, pred_valid),
        "R2": r2_score(y_valid, pred_valid),
    })

ridge_results = pd.DataFrame(ridge_results).sort_values("Log RMSE").reset_index(drop=True)

best_ridge_alpha = 16
improved_model = make_ridge_model(alpha=best_ridge_alpha)
improved_model.fit(X_train, y_train_log)

pred_valid_improved_log = improved_model.predict(X_valid)
pred_valid_improved = np.maximum(np.expm1(pred_valid_improved_log), 0)

rmse_log_improved = mean_squared_error(y_valid_log, np.log1p(pred_valid_improved)) ** 0.5
rmse_improved = mean_squared_error(y_valid, pred_valid_improved) ** 0.5
mae_improved = mean_absolute_error(y_valid, pred_valid_improved)
r2_improved = r2_score(y_valid, pred_valid_improved)

print(f"Improved model: single Ridge regression, alpha={best_ridge_alpha}")
print("Improved Log RMSE:", rmse_log_improved)
print("Improved RMSE    :", rmse_improved)
print("Improved MAE     :", mae_improved)
print("Improved R2      :", r2_improved)

ridge_results


## 8. Compare models

Briefly discuss the difference between the baseline and the improved model.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Baseline Linear Regression", "Improved single Ridge regression"],
    "Log RMSE": [rmse_log_baseline, rmse_log_improved],
    "RMSE": [rmse_baseline, rmse_improved],
    "MAE": [mae_baseline, mae_improved],
    "R2": [r2_baseline, r2_improved],
})

comparison


**Write a short discussion here.**

- Which model performed better?

The improved single Ridge regression model performed better than the baseline Linear Regression model on the local validation split and on the hidden leaderboard.

- Was the improvement large or small?

The improvement was meaningful. The best single model improved the leaderboard RMSE_LOG from 0.128492 for the first submission to 0.119292.

- What might explain the difference?

Ridge regression handles the many one-hot encoded categorical variables more safely by shrinking coefficients. Robust scaling and log-transforming skewed numeric predictors also make the feature distributions easier to model. The biggest later gain came from adding two ratio features, `SFPerRoom` and `BathPerBedroom`, because they describe how efficiently the living area and bathrooms are distributed across the house. We kept the final model as one Ridge regression instead of a blend so the result is simpler and easier to explain.


## 9. Train the final model on the full student training set

Choose your final model and fit it using all available labeled data.


In [ ]:
y_log = np.log1p(y)
final_model = make_ridge_model(alpha=best_ridge_alpha)
final_model.fit(X, y_log)

test_ids = test_df[id_col].copy()
test_for_prediction = add_engineered_features(test_df)

missing_cols = [col for col in X.columns if col not in test_for_prediction.columns]
if missing_cols:
    raise ValueError(f"Missing columns in test data after preprocessing: {missing_cols}")

test_for_prediction = test_for_prediction[X.columns]

test_predictions_log = final_model.predict(test_for_prediction)
test_predictions = np.maximum(np.expm1(test_predictions_log), 0)

submission = pd.DataFrame({
    id_col: test_ids,
    "prediction": test_predictions
})

print("Submission shape:", submission.shape)
print("Any missing predictions:", submission["prediction"].isna().any())
print("Any negative predictions:", (submission["prediction"] < 0).any())
submission.head()


## 10. Save the submission file


In [19]:
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


Saved submission.csv


## 11. Submit to the leaderboard

Upload `submission.csv` to:

<https://labo5inf01090-huzhzhojtbeqqknm7duabo.streamlit.app/>

After submitting, record your score below.


**Leaderboard score(s):**

- First submission: RMSE_LOG = 0.128492
- Ridge skew submission: RMSE_LOG = 0.120441
- Ridge no-interactions / earlier Ridge attempt: RMSE_LOG = 0.120357
- KRR/poly single-model attempt: RMSE_LOG = 0.120398
- Base Ridge single-model attempt: RMSE_LOG = 0.120361
- Ridge without decade features: RMSE_LOG = 0.120244
- Ridge ratio model, alpha=15: RMSE_LOG = 0.119299
- Ridge ratio model, alpha=16: RMSE_LOG = 0.119292
- Current best single model: `submission_single_final_05_ridge_ratio_a16.csv`
- Final submitted model in this notebook: single Ridge regression, alpha=16, using `SFPerRoom` and `BathPerBedroom`


## 12. Final reflection

Write a short final reflection addressing:

- what preprocessing choices were most important
- whether feature engineering helped
- what model worked best for your group
- what you would try next if you had more time


**Write your final reflection here.**

The most important preprocessing choices were handling missing values inside the pipeline, scaling numeric predictors with RobustScaler, log-transforming skewed numeric variables, and using one-hot encoding for categorical variables instead of arbitrary numeric labels. Feature engineering helped by adding variables that summarize property size, age, remodel status, porch area, bathrooms, and amenity indicators.

Our best hidden-leaderboard improvement came from a simple single-model change: adding `SFPerRoom` and `BathPerBedroom` to a Ridge regression model. This kept the model easy to explain while improving RMSE_LOG to 0.119292. More complicated calibration and residual-correction attempts did not generalize as well, so the final notebook keeps the stronger single Ridge model rather than a blend or ensemble. With more time, the next tests would be very small single-model variations around this same recipe, especially alpha values near 16 and the possible addition of `GaragePerCar`.


## 📤 Final Deliverables Checklist

Before submitting your work, verify that you are delivering:

- [ ] completed notebook
- [ ] generated submission file
- [ ] leaderboard score recorded
- [ ] short discussion of preprocessing and model choices
- [ ] final reflection
